In [1]:
# TÍTULO: Entrenamiento del Detector de Placas (YOLOv8)
# OBJETIVO: Generar un modelo capaz de localizar placas vehiculares con alta precisión.

import torch
from ultralytics import YOLO
import os
import shutil
import glob
import matplotlib.pyplot as plt
import cv2
from datetime import datetime

# --- VERIFICACIÓN DE GPU ---
print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU Activa: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    device = 0
else:
    print("ALERTA: Usando CPU.")
    device = 'cpu'

# RUTAS
# Apuntamos al dataset que acabas de limpiar
DATA_YAML = '../../datasets/02_placas/data.yaml'
# Guardaremos los modelos en una carpeta distinta a la de autos
PROJECT_DIR = '../../models/02_placas'

PyTorch Version: 2.9.1+cu126
GPU Activa: NVIDIA GeForce RTX 4070 Ti
VRAM: 12.45 GB


In [ ]:
# Cargar pesos pre-entrenados (Transfer Learning)
model_variant = 'yolov8n.pt'
model = YOLO(model_variant)

# Nombre del experimento
run_name = f"{datetime.now().strftime('%Y%m%d')}_v8n_clean_tl_640"

print(f"Modelo: {model_variant}")
print(f"Salida: {PROJECT_DIR}/{run_name}")

Modelo: yolov8n.pt
Salida: ../../models/02_placas/20251127_v8m_clean_tl_640


In [ ]:
# Checar numpy antes de entrenamiento. 
# Verificar versión 1.26.4 para evitar error con torch.py
import numpy
print(f"Versión de Numpy: {numpy.__version__}")

In [3]:
# --- INICIAR ENTRENAMIENTO ---
results = model.train(
    data=DATA_YAML,
    project=PROJECT_DIR,
    name=run_name,
    
    # Hiperparámetros
    epochs=30,
    patience=20,        # Early Stopping
    batch=-1,           # AutoBatch
    imgsz=640,          # Tamaño estándar
    device=device,
    
    # Optimizaciones para Placas
    pretrained=True,    # Transfer learning
    optimizer='auto',
    close_mosaic=10,    # Importante para precisión de bordes al final
    
    # Aumentación (Mantenemos rotaciones leves, las placas no suelen estar de cabeza)
    degrees=10.0,       # Rotación +/- 10 grados
    fliplr=0.0,         # Desactivar espejo horizontal
    
    verbose=True,
    exist_ok=True
)

# NOTA. Existió un conflicto en la versión de numpy2.0, por lo que se instaló la versión 1.26.4

New https://pypi.org/project/ultralytics/8.3.232 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11874MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251127_v8m_cl

In [8]:
# --- RECUPERAR HIPERPARÁMETROS REALES (CORREGIDO) ---

try:
    # Intento A: Atributo directo (nombre estándar en versiones recientes)
    real_batch_size = model.trainer.batch_size
except AttributeError:
    # Intento B: Buscar dentro de los argumentos (si no se actualizó el directo)
    real_batch_size = model.trainer.args.batch

# Validación: Si sigue saliendo -1, significa que solo guardó la configuración inicial
if real_batch_size == -1:
    print("El atributo muestra -1 (AutoBatch). Revisa el log de la consola para ver el valor final calculado.")
else:
    print(f"Batch Size Real utilizado: {real_batch_size}")

# --- OPTIMIZADOR ---
try:
    # El optimizador es un objeto complejo, sacamos su nombre de la clase
    if hasattr(model.trainer, 'optimizer'):
        optimizer_name = type(model.trainer.optimizer).__name__
    else:
        # Fallback si el objeto ya se limpió de memoria
        optimizer_name = model.trainer.args.optimizer
except:
    optimizer_name = "Auto (Revisar logs)"

print(f"Optimizador: {optimizer_name}")

# --- LEARNING RATE ---
lr0 = model.trainer.args.lr0
print(f"Learning Rate inicial: {lr0}")
print("\n")
model.info(detailed=True, verbose=True)

Batch Size Real utilizado: 47
Optimizador: AdamW
Learning Rate inicial: 0.01


layer                                    name                type  gradient  parameters               shape        mu     sigma
    0                     model.0.conv.weight              Conv2d     False         432       [16, 3, 3, 3]  -0.00524     0.198        float32
    1                       model.0.bn.weight         BatchNorm2d     False          16                [16]       2.8       1.9        float32
    1                         model.0.bn.bias         BatchNorm2d     False          16                [16]     0.333      4.33        float32
    2                             model.0.act                SiLU     False           0                  []         -         -              -
    3                     model.1.conv.weight              Conv2d     False        4608      [32, 16, 3, 3]  -0.00334     0.145        float32
    4                       model.1.bn.weight         BatchNorm2d     False   

/home/roberto/miniconda3/envs/yolov8/lib/python3.11/site-packages/ultralytics/utils/torch_utils.py:332: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  f"{i:>5g}{f'{mn}.{pn}':>40}{mt:>20}{p.requires_grad!r:>10}{p.numel():>12g}{list(p.shape)!s:>20}{p.mean():>10.3g}{p.std():>10.3g}{str(p.dtype).replace('torch.', ''):>15}"


(129, 3011043, 0, 8.1941504)

In [10]:
from torchinfo import summary

# Necesitamos acceder al modelo interno de PyTorch (model.model)
# Definimos input_size=(Batch_Size, Canales, Alto, Ancho)
# Batch=1 (simulando una imagen), Canales=3 (RGB), 640x640 (Resolución YOLO)
summary(model.model, input_size=(1, 3, 640, 640))

Layer (type:depth-idx)                             Output Shape              Param #
DetectionModel                                     [1, 5, 8400]              --
├─Sequential: 1-1                                  --                        --
│    └─Conv: 2-1                                   [1, 16, 320, 320]         --
│    │    └─Conv2d: 3-1                            [1, 16, 320, 320]         (432)
│    │    └─BatchNorm2d: 3-2                       [1, 16, 320, 320]         (32)
│    └─Detect: 2-96                                --                        (recursive)
│    │    └─ModuleList: 3-118                      --                        (recursive)
│    └─Conv: 2-3                                   [1, 32, 160, 160]         --
│    │    └─Conv2d: 3-4                            [1, 32, 160, 160]         (4,608)
│    │    └─BatchNorm2d: 3-5                       [1, 32, 160, 160]         (64)
│    └─Detect: 2-96                                --                        (recursi

In [11]:
print("Evaluando en TEST set...")

# Cargar el mejor modelo generado
best_model_path = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
best_model = YOLO(best_model_path)

# Validación
metrics = best_model.val(split='test', project=PROJECT_DIR, name=f"{run_name}_eval")

print(f"\Métricas Finales:")
print(f"   mAP@50:    {metrics.box.map50:.4f}")
print(f"   mAP@50-95: {metrics.box.map:.4f}")

Evaluando en TEST set...
Ultralytics 8.3.229 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11874MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 213.3±83.7 MB/s, size: 54.1 KB)
val: Scanning /home/roberto/proyecto/datasets/02_placas/test/labels... 2663 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2663/2663 3.3Kit/s 0.8s0.0s
val: New cache created: /home/roberto/proyecto/datasets/02_placas/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 167/167 19.5it/s 8.6s0.1s
                   all       2663       3002      0.961      0.927      0.965      0.676
Speed: 0.4ms preprocess, 1.1ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /home/roberto/proyecto/models/02_placas/20251127_v8m_clean_tl_640_eval
\Métricas Finales:
   mAP@50:    0.9647
   mAP@50-95: 0.6757


In [12]:
# Cargar pesos pre-entrenados (Transfer Learning)
model_variant = 'yolov8n.pt'
model = YOLO(model_variant)

# Nombre del experimento
run_name = f"{datetime.now().strftime('%Y%m%d')}_v8n_clean_tl_1280"

print(f"Modelo: {model_variant}")
print(f"Salida: {PROJECT_DIR}/{run_name}")

Modelo: yolov8n.pt
Salida: ../../models/02_placas/20251127_v8n_clean_tl_1280


In [13]:
# --- INICIAR ENTRENAMIENTO ---
results = model.train(
    data=DATA_YAML,
    project=PROJECT_DIR,
    name=run_name,
    
    # Hiperparámetros
    epochs=30,
    patience=20,        # Early Stopping
    batch=-1,           # AutoBatch
    imgsz=1280,          # Tamaño estándar CAMBIO
    device=device,
    
    # Optimizaciones para Placas
    pretrained=True,    # Transfer learning
    optimizer='auto',
    close_mosaic=10,    # Importante para precisión de bordes al final
    
    # Aumentación (Mantenemos rotaciones leves, las placas no suelen estar de cabeza)
    degrees=10.0,       # Rotación +/- 10 grados
    fliplr=0.0,         # Desactivar espejo horizontal
    
    verbose=True,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.3.232 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.229 🚀 Python-3.11.14 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11874MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=20251127_v8n_c

KeyboardInterrupt: 

In [ ]:
# Tomar algunas imágenes del test set
test_images = glob.glob('../../datasets/02_placas/test/images/*.jpg')[:6]

if len(test_images) == 0:
    # Fallback por si son png
    test_images = glob.glob('../../datasets/02_placas/test/images/*.png')[:6]

for img_path in test_images:
    # Inferencia
    results = best_model.predict(img_path, conf=0.4) # Confianza 0.4
    
    for r in results:
        im_array = r.plot()
        plt.figure(figsize=(6,6))
        plt.imshow(cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f"Detección Placa: {os.path.basename(img_path)}")
        plt.show()

In [ ]:
PRODUCTION_PATH = '../../production_weights'
os.makedirs(PRODUCTION_PATH, exist_ok=True)

target_path = os.path.join(PRODUCTION_PATH, '02_placas_best.pt')
shutil.copy(best_model_path, target_path)

print(f"Modelo de Placas exportado a: {target_path}")